In [1]:
import sys
sys.path.append('/home/semik/projekty/noncanonical_introns/')

import pathlib
pathlib.Path().absolute()

import introns
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt
from Bio import SeqIO
from tabulate import tabulate

In [2]:
path="/mnt/archive/euglena_genomy/"
path_rev="./fastas_and_gff/reversed_fasta_gtf/"

genom_EG=path+"gracilis/gracilis_dbg2olc.fasta"
geny_EG=path+"gracilis/gracilis_stringtie_strand_informed.gtf"
geny_EG_rev=path_rev+"gracilis_stringtie_strand_informed_reversed.gtf"

genom_EH=path+'hiemalis/hiemalis_rascaf.fasta'
geny_EH=path+'hiemalis/hiemalis_stringtie_strand_informed3.gtf'
geny_EH_rev=path_rev+"hiemalis_stringtie_strand_informed3_reversed.gtf"

genom_EL=path+'longa/longa_rascaf.fasta'
geny_EL=path+'longa/longa_stringtie_strand_informed.gtf'
geny_EL_rev=path_rev+"longa_stringtie_strand_informed_reversed.gtf"

#genom_bugtest = '/home/semik/projekty/noncanonical_introns/bugtest.fasta'
#geny_bugtest = '/home/semik/projekty/noncanonical_introns/bugtest.gtf'
#geny_bugtest_rev = '/home/semik/projekty/noncanonical_introns/bugtest_reversed.gtf'
genom_bugtest = './fastas_and_gff/bugtest.fasta'
geny_bugtest = './fastas_and_gff/bugtest.gtf'
geny_bugtest_rev = './fastas_and_gff/bugtest_reversed.gtf'

#genome_bugtest, genes_bugtest, genes_bugtest_rev = introns.create_both(genom_bugtest, geny_bugtest,
#                                                               geny_bugtest_rev, 'stringtie')
#genome_EG, genes_EG, genes_EG_rev = introns.create_both(genom_EG, geny_EG, geny_EG_rev, 'stringtie')
#genome_EH, genes_EH, genes_EH_rev = introns.create_both(genom_EH, geny_EH, geny_EH_rev, 'stringtie')
#genome_EL, genes_EL, genes_EL_rev = introns.create_both(genom_EL, geny_EL, geny_EL_rev, 'stringtie')

In [4]:
#genome_EG, genes_EG, genes_EG_rev = introns.HELP_load_default_genome_genes("EG", True)

In [5]:
#genome_EH, genes_EH, genes_EH_rev = introns.HELP_load_default_genome_genes("EH", True)

In [6]:
genome_EL, genes_EL, genes_EL_rev = introns.HELP_load_default_genome_genes("EL", True)

Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 38192/38192 [01:01<00:00, 624.07it/s]


[CREATE] Genes, sequences, introns, exons created, introns' classes predicted
Creating genes, sequences, introns, exons and introns' class prediction


100%|██████████| 38192/38192 [01:02<00:00, 608.44it/s]

[CREATE] Genes, sequences, introns, exons created, introns' classes predicted


In [8]:
def liczenie_ocen_genow(genes, genes_rev, conventional_classes, nonconventional_classes, all_count):
    ulamki_niekonw = [i/all_count for i in nonconventional_classes]
    punktacja_niekonw = [1/sqrt(sum(ulamki_niekonw[:a+1])) for a in range(len(nonconventional_classes))]
    
    kolejnosc_konw = [j[0] for j in sorted(introns.conventional_class_rate().items(), key=lambda i: i[1])][1:]
    conv_classes=[conventional_classes[k-1] for k in kolejnosc_konw]
    ulamki_konw = [i/all_count for i in conv_classes]
    punktacja_konw = [1/sqrt(sum(ulamki_konw[:a+1])) for a in range(len(conv_classes))]
    #punktacja_konw = [1/sum(ulamki_konw[:a+1]) for a in range(len(conv_classes))]
    #punktacja_konw = [1/sum([sqrt(u) for u in ulamki_konw][:a+1]) for a in range(len(conv_classes))]
    zlegeny,wszystkiegeny=[],[]
    geny1, geny2= list(genes.items()), list(genes_rev.items())
    
    for gen1, gen2, in zip(geny1, geny2):
        punkty_basic,punkty_rev = 0, 0
        no_of_introns_in_gene=len(gen1[1].introns)
        for int1, int2 in zip(gen1[1].introns, gen2[1].introns):
            oceny1, oceny2 = [], []
            if int1.best_conv_var: oceny1.append(punktacja_konw[int1.best_conv_var-1])
            if int2.best_conv_var: oceny2.append(punktacja_konw[int1.best_conv_var-1])
            if int1.best_nonconv_var: oceny1.append(punktacja_niekonw[int1.best_nonconv_var-1])
            if int2.best_nonconv_var: oceny2.append(punktacja_niekonw[int2.best_nonconv_var-1])
            
            punkty_basic = max(oceny1) if oceny1 else 0
            punkty_rev= max(oceny2) if oceny2 else 0
                
        if no_of_introns_in_gene: ocena=(punkty_basic-punkty_rev)/no_of_introns_in_gene
        else: ocena=0
        if ocena<0:
            zlegeny.append((gen1[0], ocena))
        wszystkiegeny.append((gen1[1],gen2[1], ocena))
    return zlegeny,wszystkiegeny

In [ ]:
_, conventional_classes, nonconventional_classes, *_, all_count = introns.counting_introns("", genome_EL, genes_EL)
zlegeny, wszystkiegeny = liczenie_ocen_genow(genes_EL, genes_EL_rev, conventional_classes, nonconventional_classes, all_count)


          Conventional: 22758
          Nonconventional: 16503
          Both: 792
          Other: 181218
          All: 221271


In [13]:
zlegeny[:5]

[('STRG.21.1', -1.989556813676841),
 ('STRG.24.1', -0.23985669788658148),
 ('STRG.34.1', -0.7795342681313898),
 ('STRG.48.2', -0.14083397960065688),
 ('STRG.57.1', -1.5590685362627796)]

In [14]:
wszystkiegeny[:5]

[(scaffold_1 3095 3533, scaffold_1 5671 7799, 9.99708629638377),
 (scaffold_1 8524 12008, scaffold_1 12265 17334, 0.0),
 (scaffold_1 12265 17334, scaffold_1 8524 12008, 0.0),
 (scaffold_1 5671 7799, scaffold_1 3095 3533, 0.0),
 (scaffold_1 19780 23350, scaffold_1 19780 23350, 0.0)]